# Building a clean, usable dataset with a defined target (Pipeline Construction)

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

In [2]:
# Base paths
BASE_PATH = Path("../data/raw")

ECOM_PATH = BASE_PATH / "Brazilian e-Commerce"
FUNNEL_PATH = BASE_PATH / "Marketing Funnel"

In [3]:
# Create SQLite connection
conn = sqlite3.connect("../data/processed/olist.db")

In [4]:
# Load core datasets
orders = pd.read_csv(ECOM_PATH / "olist_orders_dataset.csv")
customers = pd.read_csv(ECOM_PATH / "olist_customers_dataset.csv")
reviews = pd.read_csv(ECOM_PATH / "olist_order_reviews_dataset.csv")
items = pd.read_csv(ECOM_PATH / "olist_order_items_dataset.csv")
products = pd.read_csv(ECOM_PATH / "olist_products_dataset.csv")

In [5]:
# Load marketing datasets
leads = pd.read_csv(FUNNEL_PATH / "olist_marketing_qualified_leads_dataset.csv")
deals = pd.read_csv(FUNNEL_PATH / "olist_closed_deals_dataset.csv")

In [6]:
print(orders.shape)
print(customers.shape)
print(reviews.shape)

(99441, 8)
(99441, 5)
(99224, 7)


In [7]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [8]:
# Load dataframes into SQLite database
orders.to_sql("orders", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)
reviews.to_sql("reviews", conn, if_exists="replace", index=False)
items.to_sql("items", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)

leads.to_sql("leads", conn, if_exists="replace", index=False)
deals.to_sql("deals", conn, if_exists="replace", index=False)

842

In [9]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
pd.read_sql(query, conn)

,name
0,orders
1,customers
2,reviews
3,items
4,products
5,leads
6,deals


In [10]:
orders.columns
customers.columns
reviews.columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')

The core relationships

1. Orders → Customers
orders.customer_id = customers.customer_id
2. Customers → Real person
customers.customer_unique_id = real person
3. Reviews → Orders
reviews.order_id = orders.order_id

For:
- Order-level analysis (main goal)
👉 Use order_id
- Customer behavior / repeat purchase (later)
👉 Use customer_unique_id

### The dataset distinguishes between transactional customer IDs and persistent customer identities, allowing analysis at both order and customer levels.

In [11]:
# SQL query to join orders, customers, and reviews
query = """
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score,
    r.review_comment_message,
    r.review_comment_title
FROM orders o
LEFT JOIN customers c 
    ON o.customer_id = c.customer_id
LEFT JOIN reviews r 
    ON o.order_id = r.order_id
"""

In [12]:
df = pd.read_sql(query, conn)

In [13]:
df.isnull().sum()

order_id                             0
customer_id                          0
customer_unique_id                   0
order_purchase_timestamp             0
order_delivered_customer_date     2987
order_estimated_delivery_date        0
review_score                       768
review_comment_message           59015
review_comment_title             88424
dtype: int64

1. Missing delivery date (2987) / These are likely:

- orders not delivered yet
- cancelled orders
- failed deliveries

**These are not valid “completed experiences”**

2. Missing review score (768) / These are:

- customers who didn’t leave a review

**Important nuance:**

- Not dissatisfaction
- Not satisfaction
- Just no signal!

### I restricted the dataset to completed orders with available customer feedback to ensure a reliable target variable and meaningful feature relationships.

In [14]:
# Drop rows with missing review scores
df = df.dropna(subset=["review_score"])

In [15]:
# Drop rows with missing delivery dates
df = df.dropna(subset=["order_delivered_customer_date"])

In [16]:
df.isnull().sum()
df.shape

(96359, 9)

In [17]:
# Moving to feature engeneering, we need to convert the date columns to datetime format
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"])
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"])

In [18]:
# Delivery delay
df["delivery_delay"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

In [19]:
# Target variable
df["is_dissatisfied"] = df["review_score"].apply(lambda x: 1 if x <= 2 else 0)

In [20]:
df.groupby("is_dissatisfied")["delivery_delay"].mean()

is_dissatisfied
0   -12.912736
1    -5.149879
Name: delivery_delay, dtype: float64

In [23]:
order_values = items.groupby("order_id").agg({
    "price": "sum",
    "freight_value": "sum",
    "order_item_id": "count"
}).reset_index()

order_values["order_value"] = order_values["price"] + order_values["freight_value"]
order_values.rename(columns={"order_item_id": "product_count"}, inplace=True)

order_values = order_values[["order_id", "order_value", "product_count"]]

In [25]:
df = df.merge(order_values, on="order_id", how="left")

In [26]:
df[["order_value", "product_count"]].describe()

,order_value,product_count
count,96359.000000,96359.000000
mean,159.410276,1.141689
std,217.139422,0.535353
min,9.590000,1.000000
25%,61.790000,1.000000
50%,105.080000,1.000000
75%,176.015000,1.000000
max,13664.080000,21.000000


In [27]:
df.head()
df.describe()

,order_delivered_customer_date,order_estimated_delivery_date,review_score,delivery_delay,is_dissatisfied,order_value,product_count
count,96359,96359,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000
mean,2018-01-14 06:59:06.953258240,2018-01-25 12:15:55.377286912,4.155554,-11.917797,0.128167,159.410276,1.141689
min,2016-10-11 13:46:32,2016-10-04 00:00:00,1.000000,-147.000000,0.000000,9.590000,1.000000
25%,2017-09-25 20:34:31.500000,2017-10-05 00:00:00,4.000000,-17.000000,0.000000,61.790000,1.000000
50%,2018-02-02 17:32:40,2018-02-16 00:00:00,5.000000,-12.000000,0.000000,105.080000,1.000000
75%,2018-05-15 21:08:15.500000,2018-05-28 00:00:00,5.000000,-7.000000,0.000000,176.015000,1.000000
max,2018-10-17 13:22:46,2018-10-25 00:00:00,5.000000,188.000000,1.000000,13664.080000,21.000000
std,NaN,NaN,1.285108,10.114024,0.334277,217.139422,0.535353


In [28]:
df.to_csv("../data/processed/main_dataset.csv", index=False)